# 05 / Test `SNSLMetric`

- Author of corrections: Sylvie Dagoret-Campagne
- Creation date: 2026-08-06
- Last update: 2026-08-06
- This notebook is adapted from `03_testSNCadenceMetric.ipynb`.
- It uses `UserPointsSlicer` and a patched `SNSLMetric` that works with `band`-based v5.3+ OpSim databases.
- The SN reference object used here is a lightweight debug reference. Replace it with a scientific SN reference for production work.

## 1. Overview

— **qualité de la light curve**

## 🔬 Ce que ça mesure

La capacité à reconstruire une **courbe de lumière exploitable** :

- nombre de points utiles
- couverture avant/après maximum
- multi-bande

## ⚙️ typiquement :

- fit SALT2-like
- contraintes sur :
    - t₀
    - stretch
    - couleur

## 🧠 Interprétation

👉 “Est-ce que cette SN est **scientifiquement utilisable** ?”

✔️ inclut :

- cadence
- SNR
- sampling spectral (bandes)

In [ ]:
import os
import time
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mplconfig_opsim53"))

%matplotlib inline

import rubin_sim.maf as maf
from rubin_sim.maf.stackers import BaseStacker

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

## 2. Configuration

In [ ]:
project_data_dir = Path(os.environ.get("RUBIN_SIM_DATA_DIR", "/Users/dagoret/DATA/OpSim"))
os.environ.setdefault("RUBIN_SIM_DATA_DIR", str(project_data_dir))
print(f"RUBIN_SIM_DATA_DIR = {project_data_dir}")

science_band = "r"

run_db = project_data_dir / "baseline_v5.3.5_10yrs.db"
if not run_db.exists():
    db_candidates = sorted(project_data_dir.glob("*.db"))
    if not db_candidates:
        raise FileNotFoundError(f"No Opsim database found in {project_data_dir}")
    run_db = db_candidates[0]


baseline_file = str(run_db)
print(f"Using Opsim database: {baseline_file}")

run_name = os.path.split(baseline_file)[-1].replace(".db", "")

In [ ]:
data_dir = None

if data_dir is None:
    data_dir_itself = tempfile.TemporaryDirectory(prefix="05_maf_testSNSL_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using output directory: {data_dir}")

In [ ]:
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## 3. View for undertanding SNSLMetric

In [ ]:
# Inspect the public interface if needed.
%pinfo maf.SNSLMetric

In [ ]:
# Inspect the source if you want to compare the patched logic below with the installed implementation.
%psource maf.SNSLMetric

### 8.3 Slicer with HealpixSlicer

In [ ]:
metric = maf.SNSLMetric()
# These summaries are the most useful outputs for a single-point slicer.
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]

#### 8.3.1 Define the slicer

In [ ]:
NSIDE = 16
sqlconstraint = None

In [ ]:
slicer = maf.slicers.HealpixSlicer(nside=NSIDE, use_cache=False)

#### 8.3.2 Define the bundle

In [ ]:
bundle = maf.metric_bundle.MetricBundle(
    metric,
    slicer,
    sqlconstraint,
    # stacker_list=stackers,
    summary_metrics=sn_summary,
    run_name=run_name,
)

#### 8.3.3 Define the bundle group

In [ ]:
bdict = {"snsl": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

#### 8.3.4 Run the metric

In [ ]:
start = time.perf_counter()
group.run_all()
elapsed = time.perf_counter() - start
print(f"group.run_all() finished in {elapsed:.1f} s")

#### 8.3.5 View summary statistics

In [ ]:
bundle.compute_summary_stats()
print("Summary values:")
for name, value in bundle.summary_values.items():
    print(f"  {name}: {value}")

#### 8.3.5 Optional Plots

In [ ]:
# Plotting is optional here because a UserPointsSlicer is better inspected through the numbers above.
make_plots = True

if make_plots:
    bundle.plot()
    plt.show()